# 154 — Costo, latencia, caching y capacidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.**
`costo_peticion = 1200·(3/1e6) + 400·(15/1e6) = 0.0036 + 0.0060 = 0.0096 USD`.
Mensual: `50 000 · 0.0096 = 480 USD`.

**Ejercicio 2.** Entrada: 400 tokens no cacheables a precio pleno
(`0.0012`), prefijo con acierto `0.8·800·0.3/1e6 = 0.000192`, prefijo sin
acierto `0.2·800·3/1e6 = 0.00048` → entrada = `0.001872`. Petición =
`0.001872 + 0.006 = 0.007872 USD`; mensual = `393.60 USD`. Ahorro =
`86.40 USD/mes` = **18 %**. Nota: solo se abarata la entrada; la salida domina
y no la toca ninguna caché de prefijo.

**Ejercicio 3.** Ordenadas: 380, 390, 400, 410, 420, 430, 440, 450, 3900,
4100. Promedio = 11 320/10 = **1 132 ms**; p50 = (420+430)/2 = **425 ms**;
p95 = posición ⌈9.5⌉ = 10 → **4 100 ms**. El promedio (1 132 ms) no describe a
nadie: la experiencia típica es ~425 ms y la cola es ~4 s. El SLO se define
sobre percentiles (p. ej. «p95 < 2 000 ms»), y aquí se incumpliría.

**Ejercicio 4.** `L = λ·S = 4·3 = 12` peticiones concurrentes en promedio.
Mínimo teórico: `12/6 = 2` réplicas (ρ = 1, sin margen: cualquier ráfaga
dispara la cola). Con ρ ≤ 0.6: `c ≥ λ·S/(0.6·6) = 12/3.6 = 3.33` →
**4 réplicas**. La diferencia entre 2 y 4 es el precio de la cola no lineal
cerca de saturación.


In [ ]:
result = run_lab("observability", seed=154)
assert result["kind"] == "observability"
assert result["evidence"]
show(result)


## Reflexión

1. Tu servicio reporta latencia promedio de 900 ms y los usuarios se quejan de
   lentitud: ¿qué percentiles pedirías y por qué el promedio puede ocultar el
   problema si cada sesión hace varias llamadas?
2. La caché semántica subió su tasa de aciertos del 20 % al 45 % tras bajar el
   umbral de similitud: ¿qué métrica adicional necesitas antes de celebrar el
   ahorro y cómo la medirías?
3. Si duplicas las réplicas para bajar la espera en cola, ¿qué parte del costo
   por petición NO cambia y qué palanca tendrías que tocar para reducirla?
